# style-lora Colab quickstart

런타임 → 런타임 유형 변경 → GPU(A100 권장, 없으면 T4)로 설정한 뒤 순서대로 실행.

**절대 이 txt 원문을 공개 저장소에 커밋하지 마세요.** 이 노트북은 Colab 세션에만 파일을 올립니다.

In [ ]:
!git clone https://github.com/yeou77/-.git repo
%cd repo/style-lora
!pip install -q -r requirements.txt

## 1) 원문 txt 업로드
정제된 본문만 있는 txt 2개를 업로드합니다 (작가의 말/후기 제거된 상태).

In [ ]:
from google.colab import files
uploaded = files.upload()  # book1.txt, book2.txt 선택
import shutil, os
os.makedirs('data/raw', exist_ok=True)
for name in uploaded:
    shutil.move(name, f'data/raw/{name}')
print(os.listdir('data/raw'))

## 2) 챕터/문단 분리 + 평가 후보 추출

In [ ]:
!python scripts/preprocess.py candidates

## 3) 평가 세트 직접 고르기
`data/processed/eval_candidates.jsonl`을 열어서 읽어보고, 마음에 드는 문단 id를
~30개 골라 `eval_selected.txt`에 한 줄씩 저장하세요 (이 셀은 예시 — 실제로는
왼쪽 파일 탐색기에서 jsonl을 열어 직접 골라야 합니다).

In [ ]:
import json
cands = [json.loads(l) for l in open('data/processed/eval_candidates.jsonl', encoding='utf-8')]
for c in cands[:5]:
    print(c['id'], c['tag'], c['text'][:80])
print(f'... 총 {len(cands)}개, 파일을 열어 직접 골라 eval_selected.txt를 만드세요')

In [ ]:
# eval_selected.txt 예시 자동 생성 (책 x 태그별 상위 5개 = 임시값, 실제로는 직접 고른 목록으로 교체 권장)
from collections import defaultdict
by_key = defaultdict(list)
for c in cands:
    by_key[(c['book'], c['tag'])].append(c['id'])
selected = [i for ids in by_key.values() for i in ids[:5]]
with open('eval_selected.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(selected))
print(f'{len(selected)}개 선택됨 (임시 자동 선택 — 가능하면 직접 고른 목록으로 교체)')

## 4) 학습 코퍼스 생성

In [ ]:
!python scripts/preprocess.py build --eval-ids eval_selected.txt

## 5) QLoRA 학습 (여기가 시간이 걸리는 단계)

In [ ]:
!python scripts/train_lora.py \
    --model_name beomi/Llama-3-Open-Ko-8B-Instruct-preview \
    --train_file data/processed/train_corpus.txt \
    --output_dir outputs/geoneomulnyeo-lora

## 6) base vs +lora 비교 리포트

In [ ]:
!python scripts/eval.py --lora_dir outputs/geoneomulnyeo-lora --out outputs/eval_report.md
print(open('outputs/eval_report.md', encoding='utf-8').read())

## 7) 결과 저장 (중요: LoRA 가중치는 Colab 세션이 끝나면 사라집니다)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r outputs/geoneomulnyeo-lora /content/drive/MyDrive/geoneomulnyeo-lora